<a href="https://colab.research.google.com/github/laboratoriodecodigos/Colab-Python/blob/main/video3_sentadillas_mediapipe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏋️ Esta IA cuenta tus sentadillas y te corrige la postura
**Laboratorio de Códigos** · MediaPipe Pose Landmarker (Tasks API) + OpenCV

Graba un video **de perfil** (cámara de lado, cuerpo completo en cuadro) haciendo sentadillas. Sirve con CPU, pero con GPU T4 va más rápido el resto del notebook.

In [1]:
!pip install -q mediapipe

In [ ]:
import urllib.request
urllib.request.urlretrieve(
    "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task",
    "pose_landmarker_full.task")

Saving 6980132-uhd_2160_4096_30fps.mp4 to 6980132-uhd_2160_4096_30fps.mp4


In [11]:
from google.colab import files
subido = files.upload()          # sube tu video de sentadillas
VIDEO_IN = next(iter(subido))

Saving 8027449-uhd_4096_2160_25fps.mp4 to 8027449-uhd_4096_2160_25fps.mp4


## 1. La matemática: ángulo de la rodilla
Con tres puntos (cadera, rodilla, tobillo) calculamos el ángulo de la rodilla. Parado ≈ 170°, sentadilla profunda < 95°.

In [3]:
import numpy as np

def angulo(a, b, c):
    """Ángulo en b (grados) formado por los puntos a-b-c."""
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba, bc = a - b, c - b
    cos = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))

def inclinacion_torso(hombro, cadera):
    """Grados del torso respecto a la vertical (0 = espalda recta)."""
    dx, dy = hombro[0] - cadera[0], cadera[1] - hombro[1]
    return abs(np.degrees(np.arctan2(dx, dy)))

print(angulo((0, 0), (0, 1), (0, 2)))   # 180: pierna recta
print(angulo((1, 1), (0, 1), (0, 2)))   # 90: rodilla doblada

179.9189715653102
90.0


## 2. Contador + corrección
- **Arriba**: rodilla > 160°. **Abajo**: rodilla < 95°.
- Una repetición cuenta al volver arriba **solo si bajaste lo suficiente**; si no, aparece *"Baja más"*.
- Si el torso se inclina más de 50°, aparece *"Endereza la espalda"*.

In [13]:
import cv2, mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

ARRIBA, ABAJO, MAX_TORSO = 160, 95, 50
# Índices de MediaPipe: (hombro, cadera, rodilla, tobillo) de cada lado
IZQ, DER = (11, 23, 25, 27), (12, 24, 26, 28)

UMBRAL_ABAJO, UMBRAL_ARRIBA, MIN_SEPARACION = 0.35, 0.85, 0.8
reps, malas, fase, min_prof = 0, 0, "arriba", 1.0
mensaje, color_msg, cuadro = "Listo", (255, 255, 255), 0

detector = vision.PoseLandmarker.create_from_options(vision.PoseLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path="pose_landmarker_full.task"),
    running_mode=vision.RunningMode.VIDEO))

cap = cv2.VideoCapture(VIDEO_IN)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter("sentadillas_raw.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

reps, malas, fase, min_rodilla = 0, 0, "arriba", 180
mensaje, color_msg, cuadro = "Listo", (255, 255, 255), 0

while True:
    ok, frame = cap.read()
    if not ok:
        break
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    res = detector.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb),
                                    int(cuadro * 1000 / fps))
    cuadro += 1

    if res.pose_landmarks:
        lm = res.pose_landmarks[0]
        P = lambda i: (lm[i].x * W, lm[i].y * H)
        cad_i, cad_d, rod_i, rod_d, tob_i, tob_d = P(23), P(24), P(25), P(26), P(27), P(28)

        # 1) Profundidad: 1 = parado, 0 = cadera a la altura de la rodilla
        prof = np.mean([(r[1] - c[1]) / (t[1] - r[1] + 1e-6)
                        for c, r, t in ((cad_i, rod_i, tob_i), (cad_d, rod_d, tob_d))])
        # 2) Separación rodillas / tobillos: < 0.8 = rodillas cerrándose
        separacion = abs(rod_i[0] - rod_d[0]) / (abs(tob_i[0] - tob_d[0]) + 1e-6)
        min_prof = min(min_prof, prof)

        if fase == "arriba" and prof < UMBRAL_ARRIBA - 0.15:
            fase = "bajando"
        if fase == "bajando" and prof > UMBRAL_ARRIBA:
            if min_prof <= UMBRAL_ABAJO:
                reps += 1; mensaje, color_msg = "Bien!", (80, 255, 80)
            else:
                malas += 1; mensaje, color_msg = "Baja mas", (0, 200, 255)
            fase, min_prof = "arriba", 1.0
        if fase == "bajando" and separacion < MIN_SEPARACION:
            mensaje, color_msg = "Rodillas hacia afuera", (60, 60, 255)

        # Dibujo de ambas piernas
        for pierna in ((cad_i, rod_i, tob_i), (cad_d, rod_d, tob_d)):
            pts = [tuple(map(int, p)) for p in pierna]
            cv2.line(frame, pts[0], pts[1], (0, 255, 120), 6)
            cv2.line(frame, pts[1], pts[2], (0, 255, 120), 6)
            for p in pts:
                cv2.circle(frame, p, 10, (255, 80, 80), -1)
        cv2.putText(frame, f"Prof: {prof:.2f}", (20, 170),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 255, 255), 3)
    # panel
    cv2.rectangle(frame, (0, 0), (W, 130), (30, 30, 30), -1)
    cv2.putText(frame, f"Sentadillas: {reps}", (20, 55), cv2.FONT_HERSHEY_SIMPLEX, 1.6, (255, 255, 255), 4)
    cv2.putText(frame, f"Incompletas: {malas}", (20, 105), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (180, 180, 180), 2)
    cv2.putText(frame, mensaje, (W // 2, 80), cv2.FONT_HERSHEY_SIMPLEX, 1.4, color_msg, 4)
    out.write(frame)

cap.release(); out.release()
print(f"Sentadillas buenas: {reps}  |  Incompletas: {malas}")

Sentadillas buenas: 5  |  Incompletas: 0


In [14]:
SALIDA = "sentadillas_ia.mp4"
!ffmpeg -y -loglevel error -i sentadillas_raw.mp4 -c:v libx264 -preset veryfast -pix_fmt yuv420p {SALIDA}
from base64 import b64encode
from IPython.display import HTML
datos = b64encode(open(SALIDA, "rb").read()).decode()
HTML(f'<video width=480 controls autoplay loop><source src="data:video/mp4;base64,{datos}" type="video/mp4"></video>')

In [ ]:
files.download(SALIDA)

**Para ajustar:** si cuenta de más o de menos, mueve `ARRIBA` / `ABAJO` 5–10°. Si tu cámara no está de perfil, el ángulo sale menor y conviene subir `ABAJO`.